# Extra seeds at N = 1000 for six main-grid configurations

**Paper.** Zero-Shot Quantum Kernel Selection on Foundation Models via Geometric Alignment (TMLR submission).
**Purpose.** The main grid was run once, with random seed 42. This notebook reruns six of the 18 main-grid
configurations under two further seeds, 7 and 123, so that the paper can report how much the utility gap and
the kernel-target alignment move when only the random choices change. It answers the reviewer question
"is a single seed enough?" at the paper's actual sample size.

**Configurations (two per domain, chosen to span the range of gaps).**

| Tag | Dataset | Encoder | Seed-42 gap | Original notebook this reproduces |
|---|---|---|---|---|
| `sst2_bert` | SST-2 | BERT | -0.051 | `week2_embedding_pipeline_n1000.ipynb` + `week3_quantum_kernel_n1000_autosave.ipynb` |
| `agnews_roberta` | AG News | RoBERTa | +0.000 | `week2_agnews_roberta.ipynb` + `week3_agnews_roberta.ipynb` |
| `cifar10` | CIFAR-10 (airplane vs automobile) | CLIP | +0.000 | `week2c_cifar10_clip_autosave.ipynb` + `week3c_cifar10_clip_autosave.ipynb` |
| `fashion_vit` | Fashion-MNIST (T-shirt vs trouser) | ViT | -0.005 | `vision_vit_two_configs.ipynb` |
| `mscoco_flava` | MSCOCO image-caption matching | FLAVA | -0.060 | `multimodal_flava_two_configs.ipynb` |
| `snlive_siglip` | SNLI-VE visual entailment | SigLIP | -0.049 | `week2_siglip_joint_DRIVE.ipynb` + `week3_snlive_siglip_DRIVE.ipynb` |

**The rule followed throughout.** Every data loader, embedding function, preprocessing step, classifier, and the
quantum circuit are copied from the original notebook listed above. The only change is that the literal `42`
in those notebooks is replaced by the variable `seed`. Nothing else is tuned, added, or removed. Section 1
proves this before any expensive computation runs.

**How to run.**
1. Run Section 0 (setup) and Section 1 (seed-42 reproduction check). Section 1 takes about 15 minutes and does
   no quantum computation. **Do not continue unless every configuration prints `PASS`.**
2. Run Section 2. It runs 6 configurations x 2 seeds. Each pair takes about 5 to 15 minutes, almost all of it
   dataset download and encoding; the quantum kernels themselves take seconds. Total about 1.5 to 2 hours. It is
   resumable, so it can be run over several Colab sessions.
3. Upload the 12 files named `*_seed7_grid_record.json` and `*_seed123_grid_record.json` from the Drive folder
   `qml_project`.

Section 4 at the end explains what the records contain and how they enter the paper, and lists known issues.

## What the seed controls, configuration by configuration

The original notebooks do not all use randomness in the same places, and this notebook preserves that exactly.
This matters when reading the results: where the candidate pool is fixed, the three seeds resample *within* the
same pool of examples, so their spread will be somewhat smaller than for configurations whose pool itself is
redrawn.

| Tag | Candidate pool | Steps controlled by the seed |
|---|---|---|
| `sst2_bert` | 1100 train / 250 validation, drawn by `shuffle(seed)` | pool draw, PCA, 1000-of-1100 and 200-of-250 draws, SVMs, bootstrap |
| `agnews_roberta` | 1100 train / 250 test, drawn by `shuffle(seed)` | pool draw, PCA, 1000-of-1100 and 200-of-250 draws, SVMs, bootstrap |
| `cifar10` | **fixed**: first 550 per class (train) and 125 per class (test) in dataset order | shuffle of the pool (`random.seed`), PCA, 1000-of-1100 and 200-of-250 draws, SVMs, bootstrap |
| `fashion_vit` | 500 per class (train) and 100 per class (test), drawn by `RandomState(seed)` | pool draw, PCA, SVMs, bootstrap (no subsampling step; the pool is exactly 1000/200) |
| `mscoco_flava` | **fixed**: first 730 images in dataset order (600 train, 130 test) | matched/mismatched caption pairing, PCA, 1000-of-1200 and 200-of-260 draws, SVMs, bootstrap |
| `snlive_siglip` | **fixed**: first 400 per class (train) and 90 per class (test) in dataset order | PCA, 1000-of-1200 and 200-of-270 draws, SVMs, bootstrap |

The PCA seed matters because scikit-learn uses a randomized solver for matrices of this shape.

---
# Section 0. Setup

In [ ]:
# Packages. The versions are not pinned in the original notebooks; see Section 4, item 2, if the
# seed-42 check reports mirrored columns.
!pip install pennylane pennylane-lightning datasets transformers torch torchvision scikit-learn numpy --quiet

import numpy as np, torch, time, os, json, shutil, random, warnings
import pennylane as qml
from PIL import Image
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModel, AutoProcessor, AutoImageProcessor,
                          CLIPModel, CLIPProcessor, FlavaProcessor, FlavaModel)
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel
import sklearn
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
import transformers
print('device:', DEVICE, '| scikit-learn', sklearn.__version__, '| pennylane', qml.__version__, '| transformers', transformers.__version__)

# Results are written to the same Drive folder the main run used.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)

def checkpoint(fname, save_fn):
    """Write a file locally, then copy it to the Drive folder."""
    save_fn(fname)
    dst = os.path.join(SAVE_DIR, os.path.basename(fname))
    if os.path.abspath(fname) != os.path.abspath(dst):
        shutil.copy(fname, dst)
    print('  saved ->', dst)

def find_file(fname):
    """Return the path of a file in the Drive folder or working directory, else None."""
    for p in [os.path.join(SAVE_DIR, fname), fname]:
        if os.path.exists(p): return p
    return None

N_QUBITS = 8
SEEDS = [7, 123]
TAGS = ['sst2_bert', 'agnews_roberta', 'cifar10', 'fashion_vit', 'mscoco_flava', 'snlive_siglip']

### Shared pieces (identical in every original notebook)

`to_features` is the preprocessing used everywhere: PCA to 8 components fitted on the training pool, then min-max
scaling to [0, pi] fitted on the training pool, with test values clipped to the same interval.
`subsample` draws the final 1000 training and 200 test points; the originals use two separate
`RandomState(42)` generators, one per split, and so does this.

In [ ]:
def to_features(train_emb, test_emb, seed):
    pca = PCA(n_components=N_QUBITS, random_state=seed).fit(train_emb)
    P_tr, P_te = pca.transform(train_emb), pca.transform(test_emb)
    scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(P_tr)
    X_tr_all = np.clip(scaler.transform(P_tr), 0, np.pi)
    X_te_all = np.clip(scaler.transform(P_te), 0, np.pi)
    return X_tr_all, X_te_all

def subsample(X_tr_all, y_tr_all, X_te_all, y_te_all, seed, n_tr=1000, n_te=200):
    idx_tr = np.random.RandomState(seed).choice(len(X_tr_all), n_tr, replace=False)
    idx_te = np.random.RandomState(seed).choice(len(X_te_all), n_te, replace=False)
    return X_tr_all[idx_tr], y_tr_all[idx_tr], X_te_all[idx_te], y_te_all[idx_te]

def l2(h):
    return h / torch.linalg.norm(h, dim=-1, keepdim=True)

---
## Loaders

Each function below reproduces one original notebook's data loading, embedding, and preprocessing. It returns
the final `X_tr (1000, 8)`, `y_tr`, `X_te (200, 8)`, `y_te`. Comments name the original notebook and flag the
places where `42` has become `seed`. Encoders are loaded once and cached.

In [ ]:
_cache = {}

# ---------------- SST-2 with BERT  (week2_embedding_pipeline_n1000.ipynb) ----------------
def bert_cls(model_name, texts, batch=32):
    # First-token final-layer representation, no normalization (masked-LM family).
    if model_name not in _cache:
        _cache[model_name] = (AutoTokenizer.from_pretrained(model_name),
                              AutoModel.from_pretrained(model_name).to(DEVICE).eval())
    tok, mdl = _cache[model_name]; out = []
    for i in range(0, len(texts), batch):
        inp = tok(texts[i:i+batch], padding=True, truncation=True, max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out.append(mdl(**inp).last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(out)

def load_sst2_bert(seed):
    ds = load_dataset('nyu-mll/glue', 'sst2')
    tr = ds['train'].shuffle(seed=seed).select(range(1100))         # was seed=42
    va = ds['validation'].shuffle(seed=seed).select(range(250))     # was seed=42
    E_tr = bert_cls('bert-base-uncased', [x['sentence'] for x in tr])
    E_va = bert_cls('bert-base-uncased', [x['sentence'] for x in va])
    X_tr_all, X_te_all = to_features(E_tr, E_va, seed)
    return subsample(X_tr_all, np.array(tr['label']), X_te_all, np.array(va['label']), seed)

# ---------------- AG News with RoBERTa  (week2_agnews_roberta.ipynb) ----------------
def load_agnews_roberta(seed):
    ds = load_dataset('fancyzhx/ag_news')
    tr = ds['train'].shuffle(seed=seed).select(range(1100))         # was seed=42
    te = ds['test'].shuffle(seed=seed).select(range(250))           # was seed=42
    E_tr = bert_cls('roberta-base', [x['text'] for x in tr])
    E_te = bert_cls('roberta-base', [x['text'] for x in te])
    X_tr_all, X_te_all = to_features(E_tr, E_te, seed)
    return subsample(X_tr_all, np.array(tr['label']), X_te_all, np.array(te['label']), seed)

# ---------------- CIFAR-10 with CLIP  (week2c_cifar10_clip_autosave.ipynb) ----------------
def clip_images(images, batch=64):
    if 'clip' not in _cache:
        _cache['clip'] = (CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32'),
                          CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval())
    proc, mdl = _cache['clip']; out = []
    for i in range(0, len(images), batch):
        inp = proc(images=images[i:i+batch], return_tensors='pt', padding=True).to(DEVICE)
        with torch.no_grad():
            # Equivalent to the original get_image_features on every transformers version:
            # pooled vision output passed through the visual projection (512-dim), then L2-normalized.
            pooled = mdl.vision_model(pixel_values=inp['pixel_values']).pooler_output
            out.append(l2(mdl.visual_projection(pooled)).cpu().numpy())
    return np.vstack(out)

def load_cifar10_clip(seed):
    # Pool: the first class-0 and class-1 images in dataset order, exactly as the original. The original's
    # stopping rule uses 550 per class for both splits; its balanced_subset then keeps 550 (train) and 125 (test).
    ds = load_dataset('uoft-cs/cifar10')
    img_col = 'img' if 'img' in ds['train'].column_names else 'image'
    pools = {}
    for split in ['train', 'test']:
        imgs, lbls = [], []
        for s in ds[split]:
            if s['label'] in [0, 1]:
                imgs.append(s[img_col]); lbls.append(s['label'])
            if lbls.count(0) >= 550 and lbls.count(1) >= 550:
                break
        pools[split] = (imgs, lbls)
    random.seed(seed)                                                # was random.seed(42)
    def balanced_subset(images, labels, n_per_class):
        c0 = [(im, lb) for im, lb in zip(images, labels) if lb == 0][:n_per_class]
        c1 = [(im, lb) for im, lb in zip(images, labels) if lb == 1][:n_per_class]
        combined = c0 + c1; random.shuffle(combined)
        return [x[0] for x in combined], np.array([x[1] for x in combined])
    tr_i, tr_l = balanced_subset(*pools['train'], 550)
    te_i, te_l = balanced_subset(*pools['test'], 125)
    X_tr_all, X_te_all = to_features(clip_images(tr_i), clip_images(te_i), seed)
    return subsample(X_tr_all, tr_l, X_te_all, te_l, seed)

# ---------------- Fashion-MNIST with ViT  (vision_vit_two_configs.ipynb) ----------------
def vit_images(images, batch=64):
    if 'vit' not in _cache:
        _cache['vit'] = (AutoImageProcessor.from_pretrained('google/vit-base-patch16-224'),
                         AutoModel.from_pretrained('google/vit-base-patch16-224').to(DEVICE).eval())
    proc, mdl = _cache['vit']; out = []
    for i in range(0, len(images), batch):
        inp = proc(images=[im.convert('RGB') for im in images[i:i+batch]], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out.append(l2(mdl(**inp).last_hidden_state[:, 0, :]).cpu().numpy())
    return np.vstack(out)

def balanced_indices(labels, class_a, class_b, n_total, seed):
    labels = np.asarray(labels); rng = np.random.RandomState(seed)
    idx_a = np.where(labels == class_a)[0]; rng.shuffle(idx_a)
    idx_b = np.where(labels == class_b)[0]; rng.shuffle(idx_b)
    half = n_total // 2
    idx = np.concatenate([idx_a[:half], idx_b[:half]]); rng.shuffle(idx)
    return idx

def load_fashion_vit(seed):
    ds = load_dataset('zalando-datasets/fashion_mnist')
    col = 'image' if 'image' in ds['train'].column_names else 'img'
    ti = balanced_indices(ds['train']['label'], 0, 1, 1000, seed)   # was SEED=42
    si = balanced_indices(ds['test']['label'],  0, 1, 200,  seed)
    tr, te = ds['train'].select(ti), ds['test'].select(si)
    X_tr, X_te = to_features(vit_images(tr[col]), vit_images(te[col]), seed)
    return X_tr, np.array(tr['label']), X_te, np.array(te['label'])  # pool is already 1000/200

# ---------------- MSCOCO with FLAVA  (multimodal_flava_two_configs.ipynb) ----------------
def flava_towers(images, texts, batch=32):
    if 'flava' not in _cache:
        _cache['flava'] = (FlavaProcessor.from_pretrained('facebook/flava-full'),
                           FlavaModel.from_pretrained('facebook/flava-full').to(DEVICE).eval())
    proc, mdl = _cache['flava']; ei, et = [], []
    with torch.no_grad():
        for i in range(0, len(images), batch):
            inp = proc(images=images[i:i+batch], return_tensors='pt').to(DEVICE)
            ei.append(l2(mdl.get_image_features(pixel_values=inp['pixel_values']).last_hidden_state[:, 0, :]).cpu().numpy())
        for i in range(0, len(texts), batch):
            inp = proc(text=list(texts[i:i+batch]), return_tensors='pt', padding=True, truncation=True, max_length=77).to(DEVICE)
            et.append(l2(mdl.get_text_features(input_ids=inp['input_ids'],
                                               attention_mask=inp['attention_mask']).last_hidden_state[:, 0, :]).cpu().numpy())
    return np.vstack(ei), np.vstack(et)

def load_mscoco_flava(seed):
    # Pool: the first 780 images in dataset order (730 used), one caption each, exactly as the original.
    all_imgs, all_caps = [], []
    try:
        ds = load_dataset('nlphuji/mscoco_2014_5k_test_image_text_retrieval', split='test')
        cols = ds.column_names
        img_col = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower()][0]
        cap_col = 'caption' if 'caption' in cols else ('captions' if 'captions' in cols else [c for c in cols if 'caption' in c.lower()][0])
        for s in ds:
            cap = s[cap_col]
            if isinstance(cap, list): cap = cap[0] if len(cap) else ''
            img = s[img_col]
            if not isinstance(img, Image.Image): continue
            all_imgs.append(img.convert('RGB')); all_caps.append(str(cap))
            if len(all_imgs) >= 780: break
    except Exception as e:
        print(f'  nlphuji failed ({type(e).__name__}); lmms-lab fallback')
        for s in load_dataset('lmms-lab/COCO-Caption', split='val', streaming=True):
            img = s.get('image'); cap = s.get('answer')
            if isinstance(cap, list) and len(cap): cap = cap[0]
            if not isinstance(img, Image.Image): continue
            all_imgs.append(img.convert('RGB')); all_caps.append(str(cap))
            if len(all_imgs) >= 780: break
    assert len(all_imgs) >= 730
    rng = np.random.RandomState(seed)                                # was RandomState(42)
    def build_pairs(idxs):
        pairs = []
        for i in idxs:
            pairs.append((i, i, 1))
            j = rng.choice([k for k in idxs if k != i]); pairs.append((i, int(j), 0))
        rng.shuffle(pairs); return pairs
    train_pairs, test_pairs = build_pairs(list(range(600))), build_pairs(list(range(600, 730)))
    img_cache, txt_cache = flava_towers(all_imgs, all_caps)
    def assemble(pairs):
        return (np.array([np.concatenate([img_cache[ii], txt_cache[ci]]) for ii, ci, _ in pairs]),
                np.array([lb for _, _, lb in pairs]))
    E_tr, y_tr_all = assemble(train_pairs); E_te, y_te_all = assemble(test_pairs)
    X_tr_all, X_te_all = to_features(E_tr, E_te, seed)
    return subsample(X_tr_all, y_tr_all, X_te_all, y_te_all, seed)

# ---------------- SNLI-VE with SigLIP  (week2_siglip_joint_DRIVE.ipynb) ----------------
def siglip_towers(images, texts, batch=32):
    if 'siglip' not in _cache:
        _cache['siglip'] = (AutoProcessor.from_pretrained('google/siglip-base-patch16-224'),
                            AutoModel.from_pretrained('google/siglip-base-patch16-224').to(DEVICE).eval())
    proc, mdl = _cache['siglip']; ei, et = [], []
    with torch.no_grad():
        for i in range(0, len(images), batch):
            inp = proc(images=images[i:i+batch], return_tensors='pt').to(DEVICE)
            # Equivalent to the original get_image_features on every transformers version (pooled tower output).
            ei.append(l2(mdl.vision_model(pixel_values=inp['pixel_values']).pooler_output).cpu().numpy())
        for i in range(0, len(texts), batch):
            inp = proc(text=texts[i:i+batch], return_tensors='pt', padding='max_length', truncation=True, max_length=64).to(DEVICE)
            # Equivalent to the original get_text_features (pooled text-tower output).
            et.append(l2(mdl.text_model(**{k: v for k, v in inp.items() if k in ('input_ids', 'attention_mask', 'position_ids')}).pooler_output).cpu().numpy())
    return np.hstack([np.vstack(ei), np.vstack(et)])

def load_snlive_siglip(seed):
    # Pool: the first 400 per class (train) and 90 per class (test) in dataset order, exactly as the original.
    ds = load_dataset('sedrickkeh/snli-ve')
    cols = ds['train'].column_names
    img_col = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower() or 'img' in c.lower()][0]
    txt_col = 'hypothesis' if 'hypothesis' in cols else [c for c in cols if 'hypo' in c.lower() or 'sentence' in c.lower()][0]
    lbl_col = 'label' if 'label' in cols else [c for c in cols if 'label' in c.lower() or 'gold' in c.lower()][0]
    STR2INT = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
    def get_label(s):
        l = s[lbl_col]; return STR2INT.get(l, l) if isinstance(l, str) else l
    def load_balanced(split, n_per_class):
        imgs, txts, lbls = [], [], []; counts = {0: 0, 1: 0, 2: 0}
        for s in ds[split]:
            l = get_label(s)
            if l not in [0, 1, 2] or counts[l] >= n_per_class: continue
            img = s[img_col]
            if not isinstance(img, Image.Image): continue
            imgs.append(img.convert('RGB')); txts.append(s[txt_col]); lbls.append(l); counts[l] += 1
            if all(v >= n_per_class for v in counts.values()): break
        return imgs, txts, np.array(lbls)
    tr_i, tr_t, tr_l = load_balanced('train', 400)
    te_i, te_t, te_l = load_balanced('test' if 'test' in ds else 'validation', 90)
    X_tr_all, X_te_all = to_features(siglip_towers(tr_i, tr_t), siglip_towers(te_i, te_t), seed)
    return subsample(X_tr_all, tr_l, X_te_all, te_l, seed)

LOADERS = {'sst2_bert': load_sst2_bert, 'agnews_roberta': load_agnews_roberta, 'cifar10': load_cifar10_clip,
           'fashion_vit': load_fashion_vit, 'mscoco_flava': load_mscoco_flava, 'snlive_siglip': load_snlive_siglip}
META = {'sst2_bert': ('text', 'BERT', 'binary'), 'agnews_roberta': ('text', 'RoBERTa', 'macro'),
        'cifar10': ('vision', 'CLIP', 'binary'), 'fashion_vit': ('vision', 'ViT', 'binary'),
        'mscoco_flava': ('multimodal', 'FLAVA', 'binary'), 'snlive_siglip': ('multimodal', 'SigLIP', 'macro')}
print('loaders ready:', list(LOADERS))

---
## Quantum kernel and evaluation (identical to every original kernel notebook)

The feature map is the single-layer IQP map on 8 qubits: Hadamards, `RZ(x_i)` on each qubit, and `ZZ(x_i x_{i+1})`
on adjacent pairs implemented as CNOT, `RZ(x_i x_{i+1})`, CNOT. The kernel value is the probability of the
all-zeros outcome after applying `U(x)` then `U(x')` adjoint, computed exactly on `lightning.qubit`.

One implementation difference, with no effect on the values. The originals call `qml.kernels.kernel_matrix`,
which runs one circuit per pair of points, about 1.2 million circuits per configuration and 80 minutes. Because the
kernel is the squared overlap of two states, K(x, x') = |<phi(x')|phi(x)>|^2 with |phi(x)> = U(x)|0>, this notebook
simulates each point's state once (1,200 circuits) and forms every entry with one matrix product. On an exact
statevector simulator the two are identical to machine precision. This was checked against the saved seed-42 SST-2
kernel: all 1,000,000 training entries agree to 1e-15 and the quantum F1 is unchanged (0.655022). The self-check
below repeats the comparison against the per-pair circuit every time the notebook runs.

In [ ]:
dev = qml.device('lightning.qubit', wires=N_QUBITS)

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(dev)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])

@qml.qnode(dev)
def feature_state(x):
    iqp_feature_map(x, wires=range(N_QUBITS))
    return qml.state()

def fast_kernel(A, B=None):
    # K(x, x') = |<phi(x')|phi(x)>|^2 with |phi(x)> = U(x)|0>. One circuit per data point instead of one per pair.
    # On an exact statevector simulator this equals the per-pair circuit to machine precision (asserted below).
    S_A = np.array([feature_state(a) for a in A])
    S_B = S_A if B is None else np.array([feature_state(b) for b in B])
    return np.abs(S_A.conj() @ S_B.T) ** 2

def kernel_matrix(A, B, symmetric, label):
    K = np.zeros((len(A), len(B))); t0 = time.time()
    for i in range(len(A)):
        for j in range(i if symmetric else 0, len(B)):
            K[i, j] = quantum_kernel(A[i], B[j])
            if symmetric and i != j: K[j, i] = K[i, j]
        if (i + 1) % 100 == 0:
            print(f'    {label}: {i+1}/{len(A)} rows, {(time.time()-t0)/60:.1f} min')
    return K

def kta(K, y):
    # Centered kernel-target alignment (Cortes et al. 2012), multiclass ideal kernel.
    y = np.asarray(y); n = len(y); Y = np.zeros((n, n))
    for c in np.unique(y):
        m = (y == c).astype(float); Y += np.outer(m, m)
    H = np.eye(n) - np.ones((n, n)) / n
    Kc, Yc = H @ K @ H, H @ Y @ H
    return float(np.sum(Kc * Yc) / np.sqrt(np.sum(Kc * Kc) * np.sum(Yc * Yc)))

def concentration(K):
    mu = np.maximum(np.linalg.eigvalsh(K), 0); lam = mu / mu.sum()
    return float(np.exp(-np.sum(lam * np.log(lam + 1e-15))) / K.shape[0])

# self-check on random inputs in [0, pi]^8
_a, _b = np.random.RandomState(0).uniform(0, np.pi, (2, N_QUBITS))
assert np.isclose(quantum_kernel(_a, _a), 1.0, atol=1e-6), 'diagonal must be 1'
assert np.isclose(quantum_kernel(_a, _b), quantum_kernel(_b, _a), atol=1e-10), 'kernel must be symmetric'
_A = np.random.RandomState(1).uniform(0, np.pi, (20, N_QUBITS))
_slow = np.array([[quantum_kernel(_A[i], _A[j]) for j in range(20)] for i in range(20)])
assert np.abs(fast_kernel(_A) - _slow).max() < 1e-10, 'fast kernel must equal the per-pair circuit'
print('kernel self-check passed: unit diagonal, symmetric, fast kernel equals per-pair circuit')

---
# Section 1. Seed-42 reproduction check (required, no quantum computation)

For each configuration this runs the loader with `seed = 42` and compares the resulting `X_tr` and `X_te` with the
arrays saved by the original run in the Drive folder. If the pipeline here is truly the original one, they match.
Where the original kernel file is present, it also recomputes three rows of the test kernel and compares them with
the saved kernel, which checks the circuit end to end.

Reading the output:
* `PASS` means the features match to within 1e-4 (tiny differences come from GPU arithmetic).
* `MIRRORED` means some columns equal pi minus the saved values. This is a scikit-learn version difference in the
  PCA sign convention, not a pipeline error, but it changes the quantum kernel. See Section 4, item 2.
* `FAIL` means the pipeline does not reproduce the original. Stop and report the printed differences.

In [ ]:
REFERENCE_FILES = {   # where the seed-42 run stored its final features (first match is used)
    'sst2_bert':      ['sst2_bert_embeddings.npz', 'embeddings_week2.npz', 'kernel_matrices_week3.npz'],
    'agnews_roberta': ['agnews_roberta_embeddings.npz', 'agnews_roberta_kernel_matrices.npz'],
    'cifar10':        ['cifar10_embeddings.npz', 'cifar10_kernel_matrices.npz'],
    'fashion_vit':    ['fashion_vit_embeddings.npz', 'fashion_vit_kernel_matrices.npz'],
    'mscoco_flava':   ['mscoco_flava_embeddings.npz', 'mscoco_flava_kernel_matrices.npz'],
    'snlive_siglip':  ['snlive_siglip_embeddings.npz', 'snlive_siglip_kernel_matrices.npz'],
}
KERNEL_FILES = {'sst2_bert': 'kernel_matrices_week3.npz'}   # every other tag uses '<tag>_kernel_matrices.npz'
# SST-2 was the first configuration run; its saved kernel file has no test features. They were recovered and
# verified against the saved kernel by sst2_bert_complete.ipynb and stored in this file.
TEST_REFERENCE = {'sst2_bert': 'sst2_bert_test_labels.npz'}

def load_reference(tag):
    for f in REFERENCE_FILES[tag]:
        p = find_file(f)
        if p is None: continue
        z = np.load(p)
        for ktr, kte in [('X_tr_subset', 'X_te_subset'), ('X_tr', 'X_te')]:
            if ktr in z.files:
                R_te = z[kte] if kte in z.files else None
                if R_te is None and tag in TEST_REFERENCE and find_file(TEST_REFERENCE[tag]):
                    R_te = np.load(find_file(TEST_REFERENCE[tag]))['X_te']
                return f, z[ktr], R_te
    return None, None, None

def compare(X, R, tol=1e-4):
    d = np.abs(X - R).max(axis=0)
    dm = np.abs(X - (np.pi - R)).max(axis=0)
    if (d < tol).all(): return 'PASS', d.max()
    if ((d < tol) | (dm < tol)).all(): return 'MIRRORED', int((dm < tol).sum())
    return 'FAIL', d.max()

check = globals().get('check', {})
for tag in TAGS:
    if check.get(tag) == 'PASS':
        print(f'\n=== {tag} === already PASS in this session, skipping'); continue
    print(f'\n=== {tag} ===')
    ref_file, R_tr, R_te = load_reference(tag)
    if ref_file is None:
        print('  no seed-42 reference file found in Drive; cannot verify'); check[tag] = 'NO REFERENCE'; continue
    t0 = time.time()
    X_tr, y_tr, X_te, y_te = LOADERS[tag](42)
    status, detail = compare(X_tr, R_tr)
    msg = f'  train features vs {ref_file}: {status}'
    msg += f' (max abs diff {detail:.2e})' if status != 'MIRRORED' else f' ({detail} of 8 columns mirrored)'
    print(msg)
    if R_te is not None:
        s2, d2 = compare(X_te, R_te); print(f'  test features: {s2}')
        status = status if s2 == 'PASS' else (s2 if status == 'PASS' else status)
    kf = find_file(KERNEL_FILES.get(tag, f'{tag}_kernel_matrices.npz'))
    if status == 'PASS' and kf is not None and R_te is not None:
        K_saved = np.load(kf)['K_test_quantum']
        rows = np.array([[quantum_kernel(X_te[i], X_tr[j]) for j in range(len(X_tr))] for i in range(3)])
        kd = np.abs(rows - K_saved[:3]).max()
        print(f'  3 test-kernel rows vs saved kernel: max abs diff {kd:.2e}', '(PASS)' if kd < 1e-4 else '(FAIL)')
        if kd >= 1e-4: status = 'FAIL'
    check[tag] = status
    print(f'  {tag}: {status}  ({(time.time()-t0)/60:.1f} min)')

print('\nSUMMARY'); [print(f'  {t:<16} {s}') for t, s in check.items()]
if all(s == 'PASS' for s in check.values()):
    print('\nAll configurations reproduce the seed-42 run. Section 2 may be run.')
else:
    print('\nDo not run Section 2 for configurations that are not PASS. See Section 4.')

---
# Section 2. Run seeds 7 and 123 (about 1.5 to 2 hours, resumable)

`RUN` lists the (configuration, seed) pairs for this session. By default it includes only configurations that
passed Section 1. A pair whose record already exists in Drive is skipped, and a finished kernel matrix is saved
before evaluation, so an interrupted session loses at most the pair in progress.

In [ ]:
RUN = [(t, s) for t in TAGS for s in SEEDS if check.get(t) == 'PASS']
print(f'{len(RUN)} pairs queued:', RUN)

def f1_fn(metric):
    avg = 'binary' if metric == 'binary' else 'macro'
    return lambda yt, yp: float(f1_score(yt, yp, average=avg))

def run_pair(tag, seed):
    domain, encoder, metric = META[tag]; fscore = f1_fn(metric)
    t0 = time.time()
    X_tr, y_tr, X_te, y_te = LOADERS[tag](seed)
    assert X_tr.shape == (1000, N_QUBITS) and X_te.shape == (200, N_QUBITS)
    assert X_tr.min() >= -1e-9 and X_tr.max() <= np.pi + 1e-9
    print(f'  features ready in {(time.time()-t0)/60:.1f} min | train classes {dict(zip(*np.unique(y_tr, return_counts=True)))}')

    # classical baselines, identical settings to the main run (C = 1)
    baselines, fitted = {}, {}
    for name, clf in [('Linear', SVC(kernel='linear', C=1.0, random_state=seed)),
                      ('RBF', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=seed)),
                      ('Polynomial', SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale', random_state=seed))]:
        yp = clf.fit(X_tr, y_tr).predict(X_te); fitted[name] = clf
        baselines[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': fscore(y_te, yp)}
    best = max(baselines, key=lambda k: baselines[k]['f1']); best_f1 = baselines[best]['f1']

    # quantum kernel, checkpointed
    kfile = f'{tag}_seed{seed}_kernel_matrices.npz'; kp = find_file(kfile)
    if kp:
        z = np.load(kp); K_tr, K_te = z['K_train_quantum'], z['K_test_quantum']; print('  kernel loaded from checkpoint')
    else:
        tk = time.time()
        K_tr = fast_kernel(X_tr)            # 1000 x 1000
        K_te = fast_kernel(X_te, X_tr)      # 200 x 1000
        elapsed = (time.time() - tk) / 60
        print(f'  quantum kernels computed in {elapsed*60:.0f} s')
        checkpoint(kfile, lambda p: np.savez(p, K_train_quantum=K_tr, K_test_quantum=K_te, X_tr=X_tr, y_tr=y_tr,
                                             X_te=X_te, y_te=y_te, elapsed_min=elapsed))
    assert np.allclose(np.diag(K_tr), 1.0, atol=1e-6) and np.allclose(K_tr, K_tr.T)

    # quantum SVM, gap, and bootstrap CI (1000 test-set resamples)
    y_q = SVC(kernel='precomputed', C=1.0).fit(K_tr, y_tr).predict(K_te)
    f1_q = fscore(y_te, y_q); gap = f1_q - best_f1
    rng = np.random.RandomState(seed); y_c = fitted[best].predict(X_te); boot = []
    for _ in range(1000):
        idx = rng.randint(0, len(y_te), len(y_te))
        if len(np.unique(y_te[idx])) < 2: continue
        boot.append(fscore(y_te[idx], y_q[idx]) - fscore(y_te[idx], y_c[idx]))
    lo, hi = np.percentile(boot, [2.5, 97.5])

    K_rbf = rbf_kernel(X_tr, X_tr, gamma=1.0 / X_tr.shape[1])     # diagnostic RBF, gamma = 1/d
    off = ~np.eye(len(X_tr), dtype=bool)
    rec = {'tag': tag, 'seed': seed, 'domain': domain, 'encoder': encoder, 'metric': metric,
           'n_qubits': N_QUBITS, 'n_train': len(y_tr), 'n_test': len(y_te),
           'quantum_accuracy': float(accuracy_score(y_te, y_q)), 'quantum_f1': f1_q,
           'best_classical_f1': best_f1, 'best_classical_kernel': best, 'gap': gap,
           'ci_lo': float(lo), 'ci_hi': float(hi), 'excludes_zero': bool(lo > 0 or hi < 0),
           'kta': kta(K_tr, y_tr), 'c_kta': kta(K_rbf, y_tr), 'concentration': concentration(K_tr),
           'offdiag_quantum': float(K_tr[off].mean()), 'offdiag_rbf': float(K_rbf[off].mean()),
           'classical_baselines': baselines,
           'pipeline': 'identical to seed-42 run; seed substituted for 42', 'sklearn_version': sklearn.__version__, 'transformers_version': transformers.__version__}
    checkpoint(f'{tag}_seed{seed}_grid_record.json', lambda p: json.dump(rec, open(p, 'w'), indent=2))
    print(f'  RESULT {tag} seed {seed}: quantum F1 {f1_q:.4f} | best classical {best_f1:.4f} ({best}) | '
          f'gap {gap:+.4f} [{lo:+.4f}, {hi:+.4f}] | A_Q {rec["kta"]:.4f} | A_C {rec["c_kta"]:.4f} | '
          f'{(time.time()-t0)/60:.1f} min')
    return rec

for tag, seed in RUN:
    if find_file(f'{tag}_seed{seed}_grid_record.json'):
        print(f'{tag} seed {seed}: record exists, skipping'); continue
    print(f'\n=== {tag} / seed {seed} ===')
    run_pair(tag, seed)
print('\nqueue finished')

---
# Section 3. Summary across the three seeds

The seed-42 values below are copied from the paper's canonical results file (`grid_split.json`), which is the
source of every number in the paper. The seed-7 and seed-123 values are read from the new records. The cell prints
per-configuration means and sample standard deviations across the three seeds.

In [ ]:
SEED42 = {"agnews_roberta": {"gap": 0.000214, "c_kta": 0.380437, "kta": 0.319909}, "cifar10": {"gap": 0.0, "c_kta": 0.708294, "kta": 0.272327}, "snlive_siglip": {"gap": -0.048772, "c_kta": 0.003499, "kta": 0.009206}, "sst2_bert": {"gap": -0.0514, "c_kta": 0.038991, "kta": 0.02118}, "fashion_vit": {"gap": -0.005127, "c_kta": 0.652706, "kta": 0.249212}, "mscoco_flava": {"gap": -0.059575, "c_kta": 0.018308, "kta": 0.043153}}   # from grid_split.json (main run)
rows = []
for tag in TAGS:
    recs = [SEED42[tag]]
    for s in SEEDS:
        p = find_file(f'{tag}_seed{s}_grid_record.json')
        if p: recs.append(json.load(open(p)))
    if len(recs) < 3:
        print(f'{tag}: {len(recs)} of 3 seeds available'); continue
    g = np.array([r['gap'] for r in recs]); a = np.array([r['c_kta'] for r in recs])
    rows.append((tag, g.mean(), g.std(ddof=1), a.mean(), a.std(ddof=1), [round(float(x), 3) for x in g]))
print(f"{'tag':<16} {'gap mean':>9} {'gap sd':>7} {'A_C mean':>9} {'A_C sd':>7}   gaps (seed 42, 7, 123)")
for t, gm, gs, am, as_, gl in rows:
    print(f'{t:<16} {gm:>+9.3f} {gs:>7.3f} {am:>9.3f} {as_:>7.3f}   {gl}')
if rows: print(f'\nmean gap sd across configurations: {np.mean([r[2] for r in rows]):.4f}')

---
# Section 4. Reference notes

**1. Output files (all in `qml_project` on Drive).** For each configuration and seed:
`<tag>_seed<seed>_kernel_matrices.npz` holds the quantum Gram matrices, the features, and the labels;
`<tag>_seed<seed>_grid_record.json` holds every number the paper uses, in the same fields as the main-grid
records plus `seed`, `sklearn_version`, and `pipeline`. Only the 12 JSON records need to be uploaded.

**2. If Section 1 reports `MIRRORED`.** Newer scikit-learn releases changed the sign convention of PCA components.
A mirrored component x becomes pi - x after scaling, which changes the IQP kernel. The fix is to install the
scikit-learn version the main run used, which can be read from any Colab session history of the original
notebooks, and rerun Section 1 until every configuration is `PASS`. Do not run Section 2 on mirrored features.

**2b. Library versions and the encoder calls.** Recent `transformers` releases changed `get_image_features` and
`get_text_features` on CLIP and SigLIP to return an output object instead of a tensor. The loaders therefore compute
the same vectors directly: for CLIP, the pooled vision output through `visual_projection`; for SigLIP, the pooled
output of each tower. These are exactly what the original calls returned, and Section 1 confirms it against the
saved seed-42 features. Each record stores the `transformers` and scikit-learn versions used.

**3. If Section 1 reports `FAIL` or `NO REFERENCE`.** `NO REFERENCE` means the seed-42 feature file is not in the
Drive folder; copy it there from the original run. `FAIL` means a dataset or library has changed since the main
run (for example a dataset on the Hugging Face Hub was updated, or a different fallback source was used). The
printed maximum difference says how far off it is. Report it rather than proceeding.

**4. How this enters the paper.** The Section 3 summary gives, for six configurations, the mean and standard
deviation of the utility gap and of the classical alignment across seeds 42, 7, and 123 at N = 1000. These go into
the seed-variability table in the appendix and one sentence in the robustness section of the results, alongside
the earlier three-seed study at N = 500. The main-grid statistics in the paper stay at seed 42, as reported.

**5. Interpreting the spread.** For `cifar10`, `mscoco_flava`, and `snlive_siglip` the candidate examples are fixed
by dataset order (see the table at the top), so seeds only resample within that pool, pair captions differently,
or reorder data. Their seed spread is therefore a lower bound on full resampling variance. This is stated in the
paper wherever the numbers are used.